In [1]:
"""
Generate all unique series-parallel trees for N labeled components
and save them in a compact binary format (gzip-compressed).

Encoding: each tree is serialized in preorder traversal.
  - Internal node: 0x00 = Series, 0x01 = Parallel  (1 byte)
  - Leaf node:     0x80 | label                     (1 byte, labels 0-127)

For N=9 (19 bytes/tree, 25.6M trees): ~490 MB raw, ~100-150 MB gzip.
For N=10 (21 bytes/tree, 563M trees): ~11.8 GB raw, ~2-3 GB gzip.

Expected counts (OEIS A006351):
  N:  2    3     4      5       6       7        8         9          10
  s:  2    8    52    472   5,504  78,416  1,320,064  25,637,824  564,275,648
"""

import numpy as np
from itertools import combinations
import struct
import gzip
import sys
import time


# =====================================================================
# Compact byte encoding / decoding
# =====================================================================

def encode_tree(t):
    """Encode a tuple-tree into bytes (preorder traversal).
    
    Leaf (int):              0x80 | label
    Series  ('S', l, r):     0x00, then left, then right
    Parallel ('P', l, r):    0x01, then left, then right
    """
    buf = bytearray()
    _encode_node(t, buf)
    return bytes(buf)


def _encode_node(node, buf):
    if isinstance(node, int):
        buf.append(0x80 | node)
    else:
        op, left, right = node
        buf.append(0x01 if op == 'P' else 0x00)
        _encode_node(left, buf)
        _encode_node(right, buf)


def decode_tree(data, offset=0):
    """Decode bytes back into a tuple-tree. Returns (tree, next_offset)."""
    b = data[offset]
    if b & 0x80:  # leaf
        return (b & 0x7F), offset + 1
    else:
        op = 'P' if b == 0x01 else 'S'
        left, off = decode_tree(data, offset + 1)
        right, off = decode_tree(data, off)
        return (op, left, right), off


# =====================================================================
# File I/O — compact gzip format
# =====================================================================
# File layout (inside gzip):
#   For each N present:
#     4 bytes (uint32 LE): N
#     4 bytes (uint32 LE): count of trees
#     1 byte  (uint8):     bytes_per_tree (= 2*N - 1)
#     Then count × bytes_per_tree raw encoded trees (fixed length)

def save_compact(trees_by_n, path):
    """Save {N: [tuple_tree, ...]} to a compact gzip file."""
    with gzip.open(path, 'wb', compresslevel=6) as f:
        for N in sorted(trees_by_n.keys()):
            trees = trees_by_n[N]
            tree_size = 2 * N - 1  # every SP tree with N leaves has exactly this many bytes
            f.write(struct.pack('<IIB', N, len(trees), tree_size))
            for t in trees:
                encoded = encode_tree(t)
                assert len(encoded) == tree_size, \
                    f"Tree for N={N} encoded to {len(encoded)} bytes, expected {tree_size}"
                f.write(encoded)
    print(f"Saved to {path}")


def load_compact(path):
    """Load compact gzip file → {N: [tuple_tree, ...]}.
    
    Warning: for large N (>=9) this loads all trees into memory.
    Use load_compact_lazy() for streaming access.
    """
    result = {}
    with gzip.open(path, 'rb') as f:
        while True:
            header = f.read(9)  # 4 + 4 + 1
            if len(header) < 9:
                break
            N, count, tree_size = struct.unpack('<IIB', header)
            trees = []
            for _ in range(count):
                data = f.read(tree_size)
                tree, _ = decode_tree(data)
                trees.append(tree)
            result[N] = trees
            print(f"  Loaded N={N}: {len(trees)} trees")
    return result


def load_compact_lazy(path, target_N):
    """Generator that yields tuple-trees for a specific N without
    loading everything into memory. Skips other N values."""
    with gzip.open(path, 'rb') as f:
        while True:
            header = f.read(9)
            if len(header) < 9:
                return
            N, count, tree_size = struct.unpack('<IIB', header)
            if N == target_N:
                for _ in range(count):
                    data = f.read(tree_size)
                    tree, _ = decode_tree(data)
                    yield tree
                return
            else:
                # Skip this N entirely
                f.seek(count * tree_size, 1)


# =====================================================================
# Tree generation (same logic as before, outputs tuples)
# =====================================================================

class SPTreeGenerator:
    def __init__(self):
        self.memo = {}

    def _make_base_preds(self, N):
        num_inputs = 1 << N
        self.base_preds_packed = {}
        for i in range(N):
            pred = np.array([(j >> i) & 1 for j in range(num_inputs)], dtype=np.uint8)
            self.base_preds_packed[i] = np.packbits(pred)

    def _get_trees(self, labels_frozen):
        if labels_frozen in self.memo:
            return self.memo[labels_frozen]

        labels = sorted(labels_frozen)

        if len(labels) == 1:
            label = labels[0]
            packed = self.base_preds_packed[label]
            key = packed.tobytes()
            result = {key: (packed, label)}
            self.memo[labels_frozen] = result
            return result

        result = {}
        rest = labels[1:]

        for k in range(1, len(labels)):
            for s2_elements in combinations(rest, k):
                s2 = frozenset(s2_elements)
                s1 = labels_frozen - s2

                trees1 = self._get_trees(s1)
                trees2 = self._get_trees(s2)

                for key1, (packed1, t1) in trees1.items():
                    for key2, (packed2, t2) in trees2.items():
                        packed_s = np.bitwise_and(packed1, packed2)
                        ks = packed_s.tobytes()
                        if ks not in result:
                            result[ks] = (packed_s, ('S', t1, t2))

                        packed_p = np.bitwise_or(packed1, packed2)
                        kp = packed_p.tobytes()
                        if kp not in result:
                            result[kp] = (packed_p, ('P', t1, t2))

        self.memo[labels_frozen] = result
        return result

    def generate(self, N, verbose=True):
        if verbose:
            print(f"Generating SP trees for N={N} "
                  f"(truth table: {1 << N} bits = {(1 << N) // 8} bytes/function)...")

        self.memo = {}
        self._make_base_preds(N)

        t0 = time.time()
        full_set = frozenset(range(N))
        trees_dict = self._get_trees(full_set)
        elapsed = time.time() - t0

        tree_list = [tree_tuple for packed, tree_tuple in trees_dict.values()]

        if verbose:
            print(f"  Found {len(tree_list)} unique trees in {elapsed:.1f}s")

        self.memo = {}  # free RAM
        return tree_list


# =====================================================================
# Main — generate + save
# =====================================================================

if __name__ == "__main__":
    expected = {
        2: 2, 3: 8, 4: 52, 5: 472, 6: 5504,
        7: 78416, 8: 1320064, 9: 25637824,
        # 10: 563025408  # uncomment if you dare
    }

    max_N = 8
    out_path = "possible_trees_compact.gz"

    gen = SPTreeGenerator()

    all_trees = {}
    all_ok = True
    for N in range(2, max_N + 1):
        trees = gen.generate(N, verbose=True)
        all_trees[N] = trees

        if N in expected:
            ok = len(trees) == expected[N]
            status = "OK" if ok else f"MISMATCH (expected {expected[N]})"
            if not ok:
                all_ok = False
            print(f"  Verification: {status}")

        # Show encoded size for this N
        sample = encode_tree(trees[0])
        total_bytes = len(sample) * len(trees)
        print(f"  Encoded: {len(sample)} bytes/tree, "
              f"~{total_bytes / 1e6:.1f} MB raw for {len(trees)} trees")

    print(f"\n{'All counts match!' if all_ok else 'SOME COUNTS DO NOT MATCH'}")

    save = input(f"\nSave to {out_path}? [y/N] ").strip().lower()
    if save == 'y':
        save_compact(all_trees, out_path)
    else:
        print("Not saved.")


Generating SP trees for N=2 (truth table: 4 bits = 0 bytes/function)...
  Found 2 unique trees in 0.0s
  Verification: OK
  Encoded: 3 bytes/tree, ~0.0 MB raw for 2 trees
Generating SP trees for N=3 (truth table: 8 bits = 1 bytes/function)...
  Found 8 unique trees in 0.0s
  Verification: OK
  Encoded: 5 bytes/tree, ~0.0 MB raw for 8 trees
Generating SP trees for N=4 (truth table: 16 bits = 2 bytes/function)...
  Found 52 unique trees in 0.0s
  Verification: OK
  Encoded: 7 bytes/tree, ~0.0 MB raw for 52 trees
Generating SP trees for N=5 (truth table: 32 bits = 4 bytes/function)...
  Found 472 unique trees in 0.0s
  Verification: OK
  Encoded: 9 bytes/tree, ~0.0 MB raw for 472 trees
Generating SP trees for N=6 (truth table: 64 bits = 8 bytes/function)...
  Found 5504 unique trees in 0.0s
  Verification: OK
  Encoded: 11 bytes/tree, ~0.1 MB raw for 5504 trees
Generating SP trees for N=7 (truth table: 128 bits = 16 bytes/function)...
  Found 78416 unique trees in 0.3s
  Verification: OK



Save to possible_trees_compact.gz? [y/N]  y


Saved to possible_trees_compact.gz
